# Notebook 00: Environment Setup for LLM Fine-Tuning

## 📚 Historical Context

**Timeline**: 2017-2019 - When transformers emerged

**The Problem**: 
- Early transformer implementations (2017-2018) required complex manual setup
- Each research lab had their own training code
- Reproducibility was a nightmare
- Hardware requirements were unclear

**The Solution**:
- HuggingFace Transformers library (2018) standardized everything
- PyTorch/TensorFlow provided GPU acceleration
- Conda/pip made dependency management easier

## 🎯 What You'll Learn

1. Check GPU availability and capabilities
2. Install required libraries (PyTorch, Transformers, etc.)
3. Verify CUDA/cuDNN setup
4. Setup HuggingFace account and authentication
5. Test a simple model load to ensure everything works

## 💡 Why This Matters

- **Wrong PyTorch version** = CUDA errors during training
- **Insufficient GPU memory** = Out of Memory (OOM) errors
- **Missing dependencies** = Cryptic import errors
- **No HuggingFace auth** = Can't download gated models (Llama, Mistral)

---

## Step 1: Check System Information

First, let's understand what hardware we're working with.

In [ ]:
import sys
import platform
import subprocess

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)
print(f"Python Version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")
print(f"Architecture: {platform.machine()}")
print("=" * 60)

## Step 2: GPU Detection and CUDA Setup

### Why GPU Matters:
- **CPU training**: GPT-2 (117M params) = ~50 hours for 1 epoch
- **GPU training**: Same model = ~2 hours on RTX 3090
- **Modern LLMs** (7B+ params): Impossible on CPU, require GPU

### CUDA Versions:
- CUDA 11.8: Most compatible (PyTorch default)
- CUDA 12.1+: Latest, better performance but newer GPUs only

In [ ]:
# Check if we have a GPU available
import torch

print("=" * 60)
print("GPU INFORMATION")
print("=" * 60)

# Check PyTorch version
print(f"PyTorch Version: {torch.__version__}")

# Check CUDA availability
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    # CUDA version
    print(f"CUDA Version: {torch.version.cuda}")
    
    # Number of GPUs
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs: {num_gpus}")
    
    # Iterate through each GPU and show details
    for i in range(num_gpus):
        print(f"\n--- GPU {i} ---")
        print(f"Name: {torch.cuda.get_device_name(i)}")
        
        # Memory information
        # torch.cuda.get_device_properties gives us total memory
        props = torch.cuda.get_device_properties(i)
        total_memory_gb = props.total_memory / (1024**3)  # Convert bytes to GB
        print(f"Total Memory: {total_memory_gb:.2f} GB")
        
        # Current memory usage
        allocated_gb = torch.cuda.memory_allocated(i) / (1024**3)
        reserved_gb = torch.cuda.memory_reserved(i) / (1024**3)
        print(f"Currently Allocated: {allocated_gb:.2f} GB")
        print(f"Currently Reserved: {reserved_gb:.2f} GB")
        
        # Compute capability (important for features like Flash Attention)
        # Flash Attention requires compute capability >= 8.0 (Ampere or newer)
        compute_capability = f"{props.major}.{props.minor}"
        print(f"Compute Capability: {compute_capability}")
        
        # Check if this GPU supports key features
        supports_flash_attention = props.major >= 8
        print(f"Supports Flash Attention: {supports_flash_attention}")
        
    # Set default device
    torch.cuda.set_device(0)
    print(f"\nDefault GPU set to: GPU 0")
    
else:
    print("\n⚠️ WARNING: No GPU detected!")
    print("You can still follow along, but training will be VERY slow.")
    print("\nOptions:")
    print("1. Use Google Colab (free T4 GPU)")
    print("2. Use Kaggle Notebooks (free P100 GPU)")
    print("3. Rent cloud GPU: Lambda Labs, RunPod, Vast.ai")

print("=" * 60)

## Step 3: Install Required Libraries

### Core Libraries:
1. **torch**: PyTorch - deep learning framework
2. **transformers**: HuggingFace - pre-trained models and training utilities
3. **datasets**: HuggingFace - efficient data loading
4. **accelerate**: HuggingFace - distributed training, mixed precision
5. **peft**: Parameter-Efficient Fine-Tuning (LoRA, QLoRA)
6. **bitsandbytes**: 8-bit/4-bit quantization for QLoRA
7. **sentencepiece**: Tokenizer used by Llama, T5, etc.
8. **wandb**: Weights & Biases - experiment tracking

### Why Each Library:
- **transformers**: 99% of fine-tuning uses this
- **datasets**: 10x faster than pandas for large datasets
- **peft**: LoRA/QLoRA = most practical fine-tuning method
- **bitsandbytes**: QLoRA = train 7B models on 1x RTX 3090

In [ ]:
# Check if libraries are installed, if not, show installation command
# NOTE: Run the pip install command in your terminal, not in notebook

libraries = {
    'torch': 'PyTorch',
    'transformers': 'HuggingFace Transformers',
    'datasets': 'HuggingFace Datasets',
    'accelerate': 'HuggingFace Accelerate',
    'peft': 'Parameter-Efficient Fine-Tuning',
    'bitsandbytes': 'Quantization Library',
    'sentencepiece': 'SentencePiece Tokenizer',
    'wandb': 'Weights & Biases',
    'trl': 'Transformer Reinforcement Learning',
    'einops': 'Tensor Operations',
    'scipy': 'Scientific Computing',
    'scikit-learn': 'Machine Learning Utilities',
}

print("=" * 60)
print("CHECKING INSTALLED LIBRARIES")
print("=" * 60)

missing_libraries = []

for lib, description in libraries.items():
    try:
        if lib == 'scikit-learn':
            # scikit-learn imports as sklearn
            module = __import__('sklearn')
        else:
            module = __import__(lib)
        
        # Get version if available
        version = getattr(module, '__version__', 'unknown')
        print(f"✓ {description:35s} v{version}")
    except ImportError:
        print(f"✗ {description:35s} NOT INSTALLED")
        missing_libraries.append(lib)

print("=" * 60)

if missing_libraries:
    print("\n⚠️ Missing libraries detected!")
    print("\nRun this command in your terminal:\n")
    print(f"pip install {' '.join(missing_libraries)}")
else:
    print("\n✓ All required libraries are installed!")

## Step 4: HuggingFace Authentication

### Why You Need This:
Many models require authentication:
- **Llama 2/3**: Meta requires license agreement
- **Mistral**: Requires HF account
- **Private models**: Your own uploaded models

### How to Get Your Token:
1. Go to https://huggingface.co/settings/tokens
2. Create a token with "Read" access (or "Write" if uploading models)
3. Copy the token
4. Run the login command below

In [ ]:
from huggingface_hub import login, whoami
import os

# Check if already logged in
try:
    user_info = whoami()
    print("=" * 60)
    print("HUGGINGFACE AUTHENTICATION")
    print("=" * 60)
    print(f"✓ Already logged in as: {user_info['name']}")
    print(f"Email: {user_info.get('email', 'N/A')}")
    print("=" * 60)
except:
    print("=" * 60)
    print("HUGGINGFACE AUTHENTICATION")
    print("=" * 60)
    print("✗ Not logged in to HuggingFace")
    print("\nTo login, run this in your terminal:")
    print("huggingface-cli login")
    print("\nOr uncomment and run the next cell with your token")
    print("=" * 60)

In [ ]:
# Uncomment this to login programmatically
# NEVER commit your token to git!

# from huggingface_hub import login
# login(token="YOUR_TOKEN_HERE")

# Better: Use environment variable
# export HF_TOKEN="your_token_here"
# login(token=os.environ.get('HF_TOKEN'))

## Step 5: Test Model Loading

Let's test if everything is working by loading a small model.

### Why This Test:
- Verifies transformers library works
- Tests HuggingFace Hub connection
- Checks if model can be moved to GPU
- Ensures tokenization works

We'll use **DistilBERT** (~66M params) - small enough to load quickly, large enough to be realistic.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

print("=" * 60)
print("TESTING MODEL LOADING")
print("=" * 60)

# Model name to test with
model_name = "distilbert-base-uncased"
print(f"Loading model: {model_name}")

try:
    # Load tokenizer
    print("\n1. Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("   ✓ Tokenizer loaded successfully")
    
    # Load model
    print("\n2. Loading model...")
    model = AutoModel.from_pretrained(model_name)
    print("   ✓ Model loaded successfully")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n   Total parameters: {total_params:,}")
    print(f"   Trainable parameters: {trainable_params:,}")
    
    # Move to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n3. Moving model to {device.upper()}...")
    model = model.to(device)
    print(f"   ✓ Model on {device.upper()}")
    
    # Test tokenization and inference
    print("\n4. Testing tokenization and inference...")
    test_text = "Hello, this is a test to verify the environment setup."
    inputs = tokenizer(test_text, return_tensors="pt").to(device)
    
    # Run a forward pass
    with torch.no_grad():
        outputs = model(**inputs)
    
    print(f"   Input shape: {inputs['input_ids'].shape}")
    print(f"   Output shape: {outputs.last_hidden_state.shape}")
    print("   ✓ Inference successful")
    
    # Check memory usage if on GPU
    if device == "cuda":
        memory_allocated = torch.cuda.memory_allocated() / (1024**2)  # MB
        print(f"\n   GPU Memory Used: {memory_allocated:.2f} MB")
    
    print("\n" + "=" * 60)
    print("✓ ALL TESTS PASSED - ENVIRONMENT IS READY!")
    print("=" * 60)
    
except Exception as e:
    print(f"\n✗ Error occurred: {str(e)}")
    print("\nPlease check:")
    print("1. Internet connection (needed to download model)")
    print("2. HuggingFace authentication (if using gated models)")
    print("3. GPU drivers (if using CUDA)")
    print("=" * 60)

## Step 6: Memory Profiling Utilities

These helper functions will be useful throughout the learning journey.

### Why Memory Matters:
- **OOM errors** = most common training failure
- **Memory leaks** = gradual slowdown over epochs
- **Fragmentation** = can't allocate despite free memory

### Key Concepts:
- **Allocated memory**: Currently in use by tensors
- **Reserved memory**: Cached by PyTorch for reuse
- **Free memory**: Available on GPU

In [ ]:
def print_gpu_memory():
    """
    Print current GPU memory usage.
    Useful for debugging OOM errors.
    """
    if not torch.cuda.is_available():
        print("No GPU available")
        return
    
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        
        # Get memory stats
        allocated = torch.cuda.memory_allocated(i) / (1024**3)  # GB
        reserved = torch.cuda.memory_reserved(i) / (1024**3)    # GB
        total = torch.cuda.get_device_properties(i).total_memory / (1024**3)  # GB
        
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved:  {reserved:.2f} GB")
        print(f"  Total:     {total:.2f} GB")
        print(f"  Free:      {total - reserved:.2f} GB")
        print(f"  Usage:     {(reserved/total)*100:.1f}%")

def clear_gpu_memory():
    """
    Clear GPU memory cache.
    Call this between experiments to avoid memory accumulation.
    """
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("✓ GPU memory cache cleared")
    else:
        print("No GPU available")

# Test the utilities
print("=" * 60)
print("GPU MEMORY UTILITIES")
print("=" * 60)
print_gpu_memory()
print("\nClearing cache...")
clear_gpu_memory()
print_gpu_memory()
print("=" * 60)

## Step 7: Create Project Directory Structure

Good organization prevents headaches later.

### Recommended Structure:
```
os_ft/
├── notebooks/          # Jupyter notebooks (learning)
├── data/              # Datasets
│   ├── raw/          # Original datasets
│   ├── processed/    # Cleaned, formatted data
│   └── samples/      # Small samples for testing
├── models/           # Saved models and checkpoints
│   ├── base/        # Downloaded base models
│   ├── checkpoints/ # Training checkpoints
│   └── final/       # Final trained models
├── outputs/          # Training outputs
│   ├── logs/        # Training logs
│   └── results/     # Evaluation results
└── scripts/          # Reusable Python scripts
```

In [ ]:
import os
from pathlib import Path

# Define directory structure
base_dir = Path.cwd().parent  # Parent of notebooks directory

directories = [
    "data/raw",
    "data/processed",
    "data/samples",
    "models/base",
    "models/checkpoints",
    "models/final",
    "outputs/logs",
    "outputs/results",
    "scripts",
]

print("=" * 60)
print("CREATING PROJECT DIRECTORIES")
print("=" * 60)
print(f"Base directory: {base_dir}\n")

for directory in directories:
    dir_path = base_dir / directory
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"✓ {directory}")

# Create a .gitignore file to avoid committing large files
gitignore_content = """# Large files
*.bin
*.safetensors
*.pt
*.pth
*.onnx
*.h5

# Data files
data/raw/*
data/processed/*
!data/samples/*

# Model files
models/base/*
models/checkpoints/*
models/final/*

# Logs and outputs
outputs/logs/*
outputs/results/*
wandb/

# Python
__pycache__/
*.pyc
.ipynb_checkpoints/
.env

# Jupyter
.ipynb_checkpoints
*.ipynb_checkpoints
"""

gitignore_path = base_dir / ".gitignore"
if not gitignore_path.exists():
    with open(gitignore_path, "w") as f:
        f.write(gitignore_content)
    print(f"\n✓ Created .gitignore")

print("\n" + "=" * 60)
print("✓ PROJECT STRUCTURE CREATED")
print("=" * 60)

## 📊 Summary and Next Steps

### What We Accomplished:
✓ Verified Python and system information  
✓ Detected GPU and CUDA setup  
✓ Checked all required libraries  
✓ Authenticated with HuggingFace  
✓ Tested model loading and inference  
✓ Created memory profiling utilities  
✓ Set up project directory structure  

### Your Environment Checklist:
- [ ] GPU with 16GB+ VRAM detected (or cloud GPU setup)
- [ ] CUDA 11.8+ installed and working
- [ ] All required Python libraries installed
- [ ] HuggingFace authentication working
- [ ] Successfully loaded and ran a test model

### Common Issues and Solutions:

**1. "CUDA out of memory"**
- Solution: Use smaller batch size, gradient accumulation, or QLoRA (Stage 2)

**2. "No module named 'transformers'"**
- Solution: `pip install transformers`

**3. "Cannot find gated model"**
- Solution: Accept license on HuggingFace Hub, then login with token

**4. "CUDA version mismatch"**
- Solution: Reinstall PyTorch with correct CUDA version: https://pytorch.org/get-started/locally/

### Next Notebook:
**01_transformer_basics.ipynb** - Understanding the transformer architecture

We'll dive into:
- How attention mechanism works
- Self-attention vs cross-attention
- Encoder vs decoder architectures
- Positional encodings
- Why transformers revolutionized NLP

---

**Historical Note**: In 2017, "Attention is All You Need" introduced transformers. Before that, RNNs and LSTMs dominated NLP but were slow (sequential processing) and struggled with long sequences. Transformers enabled parallel processing and better long-range dependencies. This single paper spawned BERT, GPT, T5, and every modern LLM.